# Loan Approval Classification Modelling

This notebook builds and evaluates classification models for **Loan Approval Status** using the preprocessed classification dataset (`X_class`, `y_class`).

Models included:
- Logistic Regression
- Gaussian Naive Bayes
- K-Nearest Neighbors (k=5)

Evaluation included:
- Confusion Matrix
- Classification Report (precision, recall, f1-score)
- ROC Curve and AUC

Then the best model is selected based on Recall and AUC, and improved using GridSearchCV.

## 1. Import Libraries
Import all required packages for data handling, modelling, metrics, plotting, and tuning.

In [ ]:
# Import libraries used in this notebook.
import warnings
warnings.filterwarnings("ignore")

# Core data and plotting libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn tools for split, models, and evaluation.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score

# Use a clean style for all plots.
sns.set_style("whitegrid")

## 2. Use Preprocessed Dataset (`X_class`, `y_class`)
This cell uses existing `X_class` and `y_class` if available. If they are not in memory, it recreates the same preprocessing pipeline from the original CSV file so modelling can run end-to-end.

In [ ]:
# Check if preprocessed data already exists in memory.
preprocessed_available = ("X_class" in globals()) and ("y_class" in globals())

# Rebuild classification dataset from CSV if needed.
if not preprocessed_available:
    print("Preprocessed X_class and y_class not found in memory. Rebuilding from CSV...")
    csv_path = "loan_approval_data.csv"
    df = pd.read_csv(csv_path)

    # Helper to match column names safely.
    def normalize_name(text):
        return str(text).strip().lower().replace(" ", "").replace("_", "")

    # Remove ID-like columns.
    id_like_columns = []
    for col in df.columns:
        normalized_col = normalize_name(col)
        if normalized_col == "id" or normalized_col.endswith("id") or normalized_col in {"loanid", "applicationid"}:
            id_like_columns.append(col)
    df = df.drop(columns=id_like_columns, errors="ignore")

    # Fill missing values.
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = df.select_dtypes(exclude=[np.number]).columns.tolist()
    for col in numeric_columns:
        df[col] = df[col].fillna(df[col].mean())
    for col in categorical_columns:
        df[col] = df[col].fillna(df[col].mode(dropna=True)[0])

    # Detect target columns from possible names.
    normalized_to_original = {normalize_name(c): c for c in df.columns}
    class_target_candidates = ["loan_approval_status", "Loan Approval Status", "loan_status", "approved"]
    reg_target_candidates = ["max_allowed_loan", "Maximum Loan Amount", "maximum_loan_amount", "loan_amount_max"]

    class_target_col = None
    for candidate in class_target_candidates:
        candidate_key = normalize_name(candidate)
        if candidate_key in normalized_to_original:
            class_target_col = normalized_to_original[candidate_key]
            break

    reg_target_col = None
    for candidate in reg_target_candidates:
        candidate_key = normalize_name(candidate)
        if candidate_key in normalized_to_original:
            reg_target_col = normalized_to_original[candidate_key]
            break

    # Stop if targets are missing.
    if class_target_col is None:
        raise ValueError("Could not find Loan Approval Status target column.")
    if reg_target_col is None:
        raise ValueError("Could not find Maximum Loan Amount target column.")

    # Keep feature columns only for classification.
    selected_feature_columns = [
        col for col in df.columns
        if col not in {class_target_col, reg_target_col}
    ]

    X_class = df[selected_feature_columns].copy()
    y_class = df[class_target_col].copy()
    X_class = pd.get_dummies(X_class, drop_first=False)

    # Encode target if it is not numeric.
    if not pd.api.types.is_numeric_dtype(y_class):
        label_encoder = LabelEncoder()
        y_class = pd.Series(label_encoder.fit_transform(y_class), name=class_target_col)

# Reuse existing data if already available.
if preprocessed_available:
    print("Using existing preprocessed X_class and y_class from memory.")

# Reset index for clean alignment.
X_class = pd.DataFrame(X_class).reset_index(drop=True)
y_class = pd.Series(y_class).reset_index(drop=True)

# Quick checks for shape and class distribution.
print("X_class shape:", X_class.shape)
print("y_class shape:", y_class.shape)
print("\nTarget value counts:")
display(y_class.value_counts(dropna=False).sort_index())

## 3. Train-Test Split
Split data into train and test sets using:
- `test_size = 0.2`
- `stratify = y_class`
- `random_state = 42`

In [ ]:
# Split data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X_class,
    y_class,
    test_size=0.2,
    stratify=y_class,
    random_state=42
)

# Print split sizes.
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 4. Build and Train Three Classification Models
Train Logistic Regression, Gaussian Naive Bayes, and KNN (`k=5`) on the same training data.

In [ ]:
# Create the three required classification models.
log_reg_model = LogisticRegression(max_iter=2000, random_state=42)
gnb_model = GaussianNB()
knn_model = KNeighborsClassifier(n_neighbors=5)

# Store models in one dictionary for easy looping.
models = {
    "Logistic Regression": log_reg_model,
    "Gaussian Naive Bayes": gnb_model,
    "K-Nearest Neighbors": knn_model
}

# Train each model on the same training data.
for model_name, model_obj in models.items():
    model_obj.fit(X_train, y_train)
    print(f"Trained: {model_name}")

## 5. Evaluate Models (Confusion Matrix, Report, ROC, AUC)
For each model, this section displays:
- Confusion Matrix
- Classification Report
- ROC Curve
- AUC Score

It also stores Recall and AUC values for model comparison.

In [ ]:
# Check if this is a binary target for ROC/AUC.
unique_classes = np.sort(np.unique(y_test))
if len(unique_classes) != 2:
    raise ValueError("This notebook expects a binary target for ROC/AUC.")

# Prepare containers for model evaluation results.
positive_class = unique_classes[-1]
evaluation_rows = []

# Evaluate each model on the same test set.
for model_name, model_obj in models.items():
    print("\n" + "=" * 80)
    print(f"Model: {model_name}")
    print("=" * 80)

    # Get predictions and class probabilities.
    y_pred = model_obj.predict(X_test)
    y_prob = model_obj.predict_proba(X_test)[:, 1]

    # Confusion matrix table and heatmap.
    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:")
    display(pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"]))

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

    # Classification report with precision, recall, and f1-score.
    report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    print("Classification Report:")
    display(pd.DataFrame(report_dict).transpose().round(4))

    # ROC curve and AUC score.
    fpr, tpr, _ = roc_curve(y_test, y_prob, pos_label=positive_class)
    auc_value = roc_auc_score(y_test, y_prob)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC = {auc_value:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.title(f"ROC Curve - {model_name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    # Save key metrics for final comparison.
    positive_recall = report_dict[str(int(positive_class))]["recall"] if str(int(positive_class)) in report_dict else report_dict[str(positive_class)]["recall"]
    evaluation_rows.append({
        "Model": model_name,
        "Recall": positive_recall,
        "AUC": auc_value
    })

## 6. Compare Models by Recall and AUC
This table compares all models based on Recall and AUC (important for identifying risky clients).

In [ ]:
# Build and sort the comparison table by Recall then AUC.
comparison_df = pd.DataFrame(evaluation_rows)
comparison_df = comparison_df.sort_values(by=["Recall", "AUC"], ascending=False).reset_index(drop=True)

# Display final model comparison.
print("Model Comparison (sorted by Recall then AUC):")
display(comparison_df.round(4))

## 7. Final Output Summary
This notebook has completed the required classification tasks with the same test split for all models and clean outputs for screenshots.

In [ ]:
# Pick the best model from the sorted comparison table.
best_model_name = comparison_df.loc[0, "Model"]
print("Best model by Recall then AUC:", best_model_name)
print("\nFinal comparison table:")
display(comparison_df.round(4))

## 8. Final Notes
This notebook has:

- Trained 3 classification models on the same train-test split
- Evaluated each model with confusion matrix, classification report, ROC curve, and AUC
- Stored results in a clean comparison table based on Recall and AUC

All outputs are displayed inline for coursework screenshots.